## Setup

In [ ]:
import os
import numpy as np
import optuna
import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics import average_precision_score
import cupy as cp

from src.py_src.models import GatekeeperModel, GreatFilterModel, Specialist910Model, SpecialistMXModel
import joblib

In [ ]:
load_dotenv()

XRAY_SLIDED_PATH = os.path.join(os.getenv("SLIDED_PATH"), "xray_slided.parquet")

xray_slided_df = pd.read_parquet(XRAY_SLIDED_PATH)

target_class = 'target_class_in_24h'
target_flux = 'target_flux_in_24h'
metadata_cols = ['run_id', 'time']

features = [col for col in xray_slided_df.columns if col not in metadata_cols + [target_class, target_flux]]

# Isolamento conceitual das features (X) e do alvo bruto (y)
X = xray_slided_df[features]
y = xray_slided_df[target_class]

## Preparing Data (Block-Chronological Split) - Modificado para Especialista MX

In [ ]:
cols_to_keep = features + ['time', target_class, target_flux]
specialist_mx_pool = xray_slided_df[cols_to_keep].copy()

def block_chronological_split(df, time_col, target_col, flux_col, _train_years, _val_years, _test_years, purge_hours=24):
    """
    Divide os dados cronologicamente e aplica Purga Simétrica de 48h.
    
    MUDANÇA CRÍTICA DE TARGET:
    Para o Specialist MX, o objetivo é separar a Classe M da raríssima Classe X.
    Portanto: 0 = Impacto Moderado/Severo (< X), 1 = Ameaça Catastrófica (>= X).
    No mapeamento: X=5.
    """
    df = df.sort_values(time_col).reset_index(drop=True).copy()
    df['year'] = df[time_col].dt.year

    df['split'] = 'none'
    df.loc[df['year'].isin(_train_years), 'split'] = 'train'
    df.loc[df['year'].isin(_val_years), 'split'] = 'val'
    df.loc[df['year'].isin(_test_years), 'split'] = 'test'

    df = df[df['split'] != 'none'].reset_index(drop=True)

    df['block_change'] = df['split'] != df['split'].shift(1)
    df.loc[0, 'block_change'] = False

    drop_indices = set()
    change_indices = df[df['block_change']].index
    purge_td = pd.Timedelta(hours=purge_hours)

    for idx in change_indices:
        transition_time = df.loc[idx, time_col]
        start_purge = transition_time - purge_td
        end_purge = transition_time + purge_td

        to_drop = df[(df[time_col] >= start_purge) & (df[time_col] < end_purge)].index
        drop_indices.update(to_drop)

    df_purged = df.drop(index=list(drop_indices)).copy()

    dict_ = {'x': {}, 'y': {}, 'flux': {}}
    cols_to_drop = [target_col, flux_col, time_col, 'year', 'split', 'block_change']

    for split_name in ['train', 'val', 'test']:
        split_df = df_purged[df_purged['split'] == split_name].copy()
        dict_['x'][split_name] = split_df.drop(columns=cols_to_drop, errors='ignore')
        # ALERTA DE TARGET (Apenas X = 1, resto = 0) para métricas de classificação
        dict_['y'][split_name] = split_df[target_col].apply(lambda lb: 1 if lb >= 5 else 0)
        dict_['flux'][split_name] = split_df[flux_col]

    return dict_

# Compartilha estritamente a mesma divisão dos modelos anteriores
train_years = [2010, 2011, 2013, 2014, 2015, 2016, 2018, 2019]
val_years = [2012, 2017]
test_years = [2020, 2021, 2022, 2023, 2024]

data = block_chronological_split(
    df=specialist_mx_pool,
    time_col='time',
    target_col=target_class,
    flux_col=target_flux,
    _train_years=train_years,
    _val_years=val_years,
    _test_years=test_years,
    purge_hours=24
)

print(f"Tamanho do Treino (Original): {len(data['x']['train'])} amostras")
print(f"Tamanho da Validação (Original): {len(data['x']['val'])} amostras")
print(f"Tamanho do Teste (Original): {len(data['x']['test'])} amostras")

## 🛑 A Tripla Cascata (Triple Filtering)

Para treinar o *Specialist MX*, os dados originais devem passar por **três barreiras**: O Gatekeeper, o Great Filter e o Specialist 910. Apenas o resíduo (Target 0 = Classe M ou inferiores que vazaram; Target 1 = Classe X) será utilizado.

In [ ]:
gatekeeper_path = os.path.join(os.getenv('GLOBAL_XRAY_FINAL_MODELS_PATH'), 'gatekeeper_v1.joblib')
gatekeeper = GatekeeperModel.load(gatekeeper_path)

gf_path = os.path.join(os.getenv('GLOBAL_XRAY_FINAL_MODELS_PATH'), 'great_filter_v1.joblib')
great_filter = GreatFilterModel.load(gf_path)

s910_path = os.path.join(os.getenv('GLOBAL_XRAY_FINAL_MODELS_PATH'), 'specialist_910_v1.joblib')
spec_910 = Specialist910Model.load(s910_path)

print(f"Limiares: GK={gatekeeper.threshold:.4f} | GF={great_filter.threshold:.4f} | S910={spec_910.threshold:.4f}\n")

In [ ]:
def filter_by_triple_cascade(x_raw: pd.DataFrame, y_raw: pd.Series, flux_raw: pd.Series):
    """Filtra o dataset mantendo apenas instâncias aprovadas pelas TRÊS etapas anteriores."""
    mask_gk = gatekeeper.predict(x_raw) == 1
    mask_gf = great_filter.predict(x_raw) == 1
    mask_s910 = spec_910.predict(x_raw) == 1
    
    # Sobreviventes das três barreiras
    mask = mask_gk & mask_gf & mask_s910
    
    return x_raw[mask].copy(), y_raw[mask].copy(), flux_raw[mask].copy()

X_train_smx, y_train_bin_smx, flux_train_smx = filter_by_triple_cascade(data['x']['train'], data['y']['train'], data['flux']['train'])
X_val_smx, y_val_bin_smx, flux_val_smx = filter_by_triple_cascade(data['x']['val'], data['y']['val'], data['flux']['val'])
X_test_smx, y_test_bin_smx, flux_test_smx = filter_by_triple_cascade(data['x']['test'], data['y']['test'], data['flux']['test'])

# PREPARAÇÃO DO TARGET DE REGRESSÃO: O modelo treina para prever o LOG10 DO FLUXO.
# Aplicamos um limite inferior (1e-9 W/m²) para representar o ruído de fundo do Sol e evitar log10(0) = -inf
BASELINE_FLUX = 1e-9

log_flux_train_smx = np.log10(np.clip(flux_train_smx, a_min=BASELINE_FLUX, a_max=None))
log_flux_val_smx = np.log10(np.clip(flux_val_smx, a_min=BASELINE_FLUX, a_max=None))
log_flux_test_smx = np.log10(np.clip(flux_test_smx, a_min=BASELINE_FLUX, a_max=None))

def print_funnel_report(name: str, original_y: pd.Series, survived_y: pd.Series):
    orig_total = len(original_y)
    surv_total = len(survived_y)
    red_pct = ((orig_total - surv_total) / orig_total) * 100
    
    orig_pos = (original_y == 1).sum()
    orig_neg = (original_y == 0).sum()
    surv_pos = (survived_y == 1).sum()
    surv_neg = (survived_y == 0).sum()
    
    noise_reduction = ((orig_neg - surv_neg) / orig_neg * 100) if orig_neg > 0 else 0
    signal_retention = (surv_pos / orig_pos * 100) if orig_pos > 0 else 0
    
    print(f"📊 {name.upper()} CASCADE REPORT (M vs X)")
    print(f"   Volume Total Original: {orig_total} -> Filtrado Final: {surv_total} (Redução global de {red_pct:.1f}%)")
    print(f"   Classes < X: {orig_neg} -> {surv_neg} amostras (Ruído Inferior Eliminado: {noise_reduction:.1f}%)")
    print(f"   Classes X:   {orig_pos} -> {surv_pos} amostras (Sinal Catastrófico Retido: {signal_retention:.1f}%)")
    print("-" * 75)

print("\n--- IMPACTO FINAL DA CASCATA SOBRE O SPECIALIST MX ---\n")
print_funnel_report("Treino", data['y']['train'], y_train_bin_smx)
print_funnel_report("Validação", data['y']['val'], y_val_bin_smx)
print_funnel_report("Teste", data['y']['test'], y_test_bin_smx)

## Discovery Model
Instanciamos a classe `SpecialistMXModel`. Sendo um regressor puro (XGBRegressorAdapter), ele focará na minimização do erro quadrático contínuo para encontrar as features que melhor explicam as flutuações extremas do logaritmo do fluxo.

In [ ]:
discovery_model = SpecialistMXModel(
    params={
        'objective': 'reg:squarederror',
        'n_estimators': 300,
        'learning_rate': 0.05,
        'max_depth': 5,
        'n_jobs': -1,
        'random_state': 42
    }
)

In [ ]:
# O Quick Scan vai treinar prevendo log_flux_train_smx
selected_features = discovery_model.discover_top_features(
    x=X_train_smx,
    y=log_flux_train_smx,
    cumulative_threshold=0.95
)

In [ ]:
selected_features

## Hyperparameter Tuning (Optuna - Regressão)
Nesta etapa, o XGBoost otimiza sua capacidade de regressão (`reg:squarederror`) focando em minimizar o RMSE na escala logarítmica. Nós avaliamos internamente a performance convertendo as predições contínuas de volta para pseudo-probabilidades logísticas.

In [ ]:
print("Transferindo dados para a VRAM da GPU...")

X_train_filtered = X_train_smx[selected_features].astype('float32')
X_val_filtered = X_val_smx[selected_features].astype('float32')

X_train_gpu = cp.array(X_train_filtered.values)
# IMPORTANTE: O Target da regressão na GPU
y_train_reg_gpu = cp.array(log_flux_train_smx.values.astype('float32'))

X_val_gpu = cp.array(X_val_filtered.values)
y_val_reg_gpu = cp.array(log_flux_val_smx.values.astype('float32'))

print("Transferência concluída. Dados alocados na GPU.")

In [ ]:
def objective(trial):
    # Regressores não possuem callbacks de poda nativos focados em PR-AUC para classes.
    # Otimizaremos puramente a precisão matemática da árvore.
    
    params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'n_estimators': 1000,
        'random_state': 1502,
        'n_jobs': -1,
        'device': 'cuda',
        'early_stopping_rounds': 50,

        # Regressores NÃO utilizam scale_pos_weight
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
        'gamma': trial.suggest_float('gamma', 0.0, 3.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 3, 15)
    }

    model = SpecialistMXModel(
        params=params, 
        features_to_keep=None
    )

    model.fit(
        x=X_train_gpu,
        y=y_train_reg_gpu,
        eval_set=[(X_val_gpu, y_val_reg_gpu)],
        verbose=False
    )

    # Fazemos a predição contínua (log_flux)
    y_pred_reg_raw = model.predict(X_val_gpu)
    y_pred_reg_cpu = y_pred_reg_raw.get() if hasattr(y_pred_reg_raw, 'get') else y_pred_reg_raw

    # Calculamos a Pseudo-Probabilidade (Logística Ancorada na Física -4.0)
    k = 10
    cutoff_x = -4.0
    y_pred_pseudo_prob = 1 / (1 + np.exp(-k * (y_pred_reg_cpu - cutoff_x)))
    
    # Otimizamos matematicamente em direção à melhor PR-AUC binária derivada da regressão
    pr_auc = average_precision_score(y_val_bin_smx, y_pred_pseudo_prob)

    return pr_auc

In [ ]:
study = optuna.create_study(direction='maximize')
print("\nIniciando tuning...")
study.optimize(objective, n_trials=500)

print(f"\nBest Score (PR AUC Virtual): {study.best_value:.4f}")
best_params = study.best_params

best_params.update({
    'n_estimators': 1000, 'objective': 'reg:squarederror',
    'eval_metric': 'rmse', 'random_state': 1502,
    'n_jobs': -1, 'early_stopping_rounds': 50
})

In [ ]:
final_model = SpecialistMXModel(
    params=study.best_params, 
    features_to_keep=selected_features
)

final_model.fit(
    x=X_train_smx, 
    y=log_flux_train_smx,
    verbose=True
)

## 🛑 Threshold Fixo pela Física

Diferente dos classificadores anteriores, a fronteira de decisão do *Specialist MX* não pode ser calibrada dinamicamente via `optimize_threshold`. Por definição astronômica, uma explosão de classe X é todo e qualquer evento cujo fluxo excede $10^{-4}$ W/m² (ou `-4.0` na escala logarítmica). Toda a nossa análise subsequente usará esse valor como *hard cutoff* absoluto.

## Results

In [ ]:
# ==========================================
# 1. PRÉ-COMPUTAÇÃO DE VETORES (Regressão)
# ==========================================
x_test_df = X_test_smx
y_true = y_test_bin_smx.values.astype(int)
flux_test = flux_test_smx

y_pred_cont = final_model.predict(x_test_df)

cutoff_x = -4.0
k = 10

# Converte log contínuo para classe binária restrita pela lei da física
y_pred = (y_pred_cont >= cutoff_x).astype(int)
# Converte log contínuo para probabilidade virtual a fim de alimentar as curvas ROC e PR-AUC
y_prob = 1 / (1 + np.exp(-k * (y_pred_cont - cutoff_x)))

# Dedução da Persistência adaptada à Extrema Raridade: Só prevejo X se houve um X recente.
soma_flares_passado = x_test_df['count_X_24h']
y_persistence = (soma_flares_passado > 0).astype(int).values

In [ ]:
print("--- RELATÓRIO DE CLASSIFICAÇÃO (EXTREMO TOPO) ---")
print(final_model.get_classification_report(y_true, y_pred, target_names=['Impacto Severo (< X)', 'Ameaça Catastrófica (X)']))

In [ ]:
print("\n--- MÉTRICAS ABRANGENTES ---")
comprehensive_df = final_model.get_comprehensive_metrics(y_true, y_pred, y_prob)
display(comprehensive_df)

In [ ]:
print("\n--- PR-F1 (SKILL SCORE RELATIVO) ---")
pr_f1_score = final_model.calculate_prss(y_true, y_pred, y_persistence)
print(f"PR-F1 Score: {pr_f1_score:.4f}")

In [ ]:
print("\n--- ANÁLISE AC/NC (ACTIVITY CHANGE) ---")
ac_nc_df = final_model.analyze_ac_nc_performance(y_true, y_pred, y_persistence)
display(ac_nc_df)

In [ ]:
print("\n--- DISTRIBUIÇÃO DE ERROS POR CLASSE SOLAR ---")
error_dist_df = final_model.analyze_error_distribution(x=x_test_df, y_true=y_true, flux_values=flux_test)
display(error_dist_df)

In [ ]:
print("\n--- ANÁLISE DE FLUXO (ZONAS) ---")
fig_flux, summary_flux = final_model.analyze_flux_errors(x=x_test_df, y_true=y_true, flux_values=flux_test, buffer_limits=None)
display(summary_flux)
display(fig_flux)

## Features Importance

In [ ]:
print("\n--- IMPORTÂNCIA DAS FEATURES (GAIN) ---")
features_importance = final_model.get_feature_importance()
display(features_importance.head(10))

## Export

In [ ]:
SAVE_PATH = os.getenv('GLOBAL_XRAY_FINAL_MODELS_PATH')
os.makedirs(SAVE_PATH, exist_ok=True)

final_model.save(os.path.join(SAVE_PATH, 'specialist_mx_v1.joblib'))

print(f"Modelo Specialist MX exportado com sucesso para: {SAVE_PATH}")
print(f"Total de features retidas: {len(final_model.features_to_keep)}")